In [1]:
import pynmea2
import json
from datetime import datetime, timezone


# ── 1. LOAD TEST DATA ──────────────────────────────────────────────────────────

def load_nmea_file(filepath):
    """Read raw NMEA sentences from a file, one per line."""
    with open(filepath, "r") as f:
        return [line.strip() for line in f if line.strip()]


# ── 2. PARSE SENTENCES ─────────────────────────────────────────────────────────

def parse_sentence(raw):
    """
    Try to parse a raw NMEA string.
    Returns the parsed object or None if it fails.
    """
    try:
        return pynmea2.parse(raw)
    except pynmea2.ParseError as e:
        print(f"[WARN] Could not parse: {raw!r} — {e}")
        return None


# ── 3. QUALITY ASSESSMENT ──────────────────────────────────────────────────────

def assess_gnss_quality(rmc):
    """
    Score the quality of a GNSS-derived position (0.0 = no trust, 1.0 = full trust).
    Based on fix status and HDOP (Horizontal Dilution of Precision).
    """
    # Void fix — GNSS has no valid position at all
    if rmc.status == "V":
        return 0.0

    score = 1.0

    # HDOP check — measures satellite geometry quality
    # Low HDOP = good geometry, high HDOP = poor geometry
    hdop = getattr(rmc, "horizontal_dil", None)
    if hdop:
        hdop = float(hdop)
        if hdop > 2.0:
            score -= 0.3   # noticeably degraded
        elif hdop > 1.0:
            score -= 0.1   # slightly degraded

    return round(score, 2)


# ── 4. BUILD CATL TRACK OBJECT ─────────────────────────────────────────────────

def build_track(rmc, hdt=None):
    """
    Map parsed NMEA data into a CATL-compatible track dictionary.
    This is what gets sent to Person B's world model.
    """
    confidence = assess_gnss_quality(rmc)

    track = {
        "trackId": f"TRK-{rmc.timestamp}",
        "type": "SURFACE",
        "source": "NMEA_GNSS",
        "timestamp": rmc.datetime.isoformat() if rmc.status == "A" else None,
        "position": {
            "lat": rmc.latitude if rmc.status == "A" else None,
            "lon": rmc.longitude if rmc.status == "A" else None,
        },
        "velocity": {
            "speedKnots": float(rmc.spd_over_grnd) if rmc.spd_over_grnd else None,
            "courseOverGround": float(rmc.true_course) if rmc.true_course else None,
            "trueHeading": float(hdt.heading) if hdt else None,
        },
        "pntConfidence": confidence,
        "gnssStatus": "A" if rmc.status == "A" else "V",
    }

    return track


# ── 5. PROCESS THE FILE ────────────────────────────────────────────────────────

def process_nmea_stream(filepath):
    """
    Read a .nmea file, pair up RMC + HDT sentences,
    and produce a list of track objects.
    """
    sentences = load_nmea_file(filepath)

    tracks = []
    last_hdt = None  # HDT doesn't always arrive paired with RMC

    for raw in sentences:
        msg = parse_sentence(raw)
        if msg is None:
            continue

        # Hold onto the latest HDT so we can attach it to the next RMC
        if isinstance(msg, pynmea2.HDT):
            last_hdt = msg

        elif isinstance(msg, pynmea2.RMC):
            track = build_track(msg, hdt=last_hdt)
            tracks.append(track)
            print(json.dumps(track, indent=2, default=str))

    return tracks


# ── 6. ENTRY POINT ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    results = process_nmea_stream("mock_data.txt")
    print(f"\n✓ Processed {len(results)} track updates")

ModuleNotFoundError: No module named 'pynmea2'